Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Running Umbrella Sampling

Umbrella sampling is a technique that applies artificial "umbrella" potentials to the system to keep it at a desired location, enabling sampling of structures that are rare events in normal molecular simulations, such as high-energy unstable states (transition states, etc.).

In the previous steps, initial structures along the reaction coordinate (Cu atom z-coordinate) were prepared.
In this notebook, umbrella sampling MD simulations with harmonic potential restraints will be run in parallel using `joblib` for each initial structure.

## Step 1. Import Libraries and Environment Setup

Load the required libraries and set environment variables for using PLUMED from Python.

In [ ]:
# Install packages as needed
#!pip install joblib
#!pip install plumed

In [ ]:
# ============================================================
# 1. Environment & Imports
# ============================================================

import os
import sys
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.constraints import FixAtoms,Atoms

# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. Calculation Parameter Settings

Set the simulation temperature, duration, and the reaction coordinate range for umbrella sampling.
Also specify the parallelization settings for `joblib`.

* **N_JOBS**: Number of calculations to run simultaneously.
* **colvars**: List of target distances for umbrella sampling (e.g., 10.2Å, 10.4Å, ...).

In [ ]:
# ============================================================
# 2. Parameters & Settings
# ============================================================

# PFP Settings
CALC_MODE     = "PBE_PLUS_D3"
METHOD_TYPE   = "PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings (Parallelization)
N_JOBS  = 10           # Number of simultaneous jobs (adjust to your environment)
VERBOSE = 10           # Progress display verbosity
BACKEND = "threading"  # "threading" recommended when using Matlantis

# MD Settings
TEMPERATURE  = 375.0        # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 50_000       # Steps per window
LOG_INTERVAL = 100

# Reaction Coordinate (Umbrella Windows)
# 10.2Å to 18.0Å in 0.2Å increments (40 windows)
cv_restraint = np.linspace(10.2, 18.0, 40)

print(f"Target Windows: {len(cv_restraint)}")
print(f"Values: {cv_restraint}")

## Step 3. Define Function to Run Umbrella Sampling for Each Window
To enable parallelization, the entire process of receiving a single restraint position (cv_at), running MD, and saving files is wrapped in a function (run_us_window).

#### Notes
- The `Estimator` (PFP calculator) is created inside this function (to prevent issues during parallelization).
- Cu bottom layer fixation (`FixAtoms`) is also configured here.
- The PLUMED configuration string is dynamically created based on the received `z_at`.

#### Configuration Items
| Item | Description |
|:-----|:-----|
|`UNITS`| Unit system settings. Here, length is set to Å and energy to eV.|
|`POSITION`| Set atom position as the collective variable.<br><br> **(Note)**: PLUMED atom indices are **1-based**. Since ASE (Python) uses **0-based** indexing, specify `ASE index + 1` when setting the value. |
|`RESTRAINT` | Apply harmonic restraint. <br><br> - `ARG`: Specify the target collective variable as argument.<br> - `KAPPA`: Specify the spring constant. <br> - `AT`: Specify the restraint position.|
|`PRINT` | Log output settings. Outputs CV values and bias amounts to the `COLVAR` file.|

In [ ]:
# ============================================================
# 3. Parallel Function Definition
# ============================================================
def run_us_window(cv_at):
    """
    Function to run umbrella sampling at specified reaction coordinate z_at (Z-direction distance)
    """
    s_time = perf_counter()
    
    # String conversion (for filenames and PLUMED input)
    cv_at_str = f"{cv_at:.2f}"
    
    # --------------------------------------------------------
    # 1. Prepare Directories
    # --------------------------------------------------------
    out_dir = f"./output/04_umbrella_sampling/cv_at_{cv_at_str}"
    os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------------
    # 2. Prepare Calculator
    # --------------------------------------------------------
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION
    )
    calculator = ASECalculator(estimator)

    # --------------------------------------------------------
    # 3. Prepare Atoms & Constraints
    # --------------------------------------------------------
    # Load from inputs, fall back to assets if not found
    input_xyz = f'./inputs/initial_rc_{cv_at_str}.xyz'
    if not os.path.exists(input_xyz):
        input_xyz = f'./assets/05_umbrella_sampling/initial_rc_{cv_at_str}.xyz'

    if not os.path.exists(input_xyz):
        return f"Error: Input file not found for cv_at={cv_at_str}"

    atoms = read(input_xyz)

    # Fix bottom 1st and 2nd Cu layers (fix atoms with z below threshold)
    thresh = 4.0
    constraint = FixAtoms(mask=atoms.positions[:, 2] < thresh)
    atoms.set_constraint(constraint)

    # --------------------------------------------------------
    # 4. PLUMED Settings
    # --------------------------------------------------------
    # Umbrella potential settings
    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",

        # Get the coordinates of the restrained atom (PLUMED uses 1-based indexing)
        f"pos: POSITION ATOM=268",

        # Umbrella potential (Harmonic Restraint)
        f"restraint-z: RESTRAINT ARG=pos.z KAPPA=2.5 AT={cv_at_str}",

        # Output settings
        f"PRINT STRIDE={LOG_INTERVAL} ARG=pos.z,restraint-z.bias,restraint-z.force2 FILE={out_dir}/COLVAR_{cv_at_str}",
        "FLUSH STRIDE=1000"
    ]

    # Connect PLUMED Calculator
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE
    )

    # --------------------------------------------------------
    # 5. MD Simulation Setup
    # --------------------------------------------------------
    # Set initial velocities
    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)

    # Set up Langevin Dynamics
    dyn = Langevin(
        atoms, 
        TIMESTEP, 
        temperature_K=TEMPERATURE, 
        friction=0.002/units.fs, 
        trajectory=f'{out_dir}/md-dyn.traj', 
        logfile=f'{out_dir}/md-dyn.log', 
        loginterval=LOG_INTERVAL
    )

    # --------------------------------------------------------
    # 6. Run Execution
    # --------------------------------------------------------
    try:
        dyn.run(TOTAL_STEPS)

        # Save restart files
        write(f'{out_dir}/md-dyn-restart.cif', atoms)
        write(f'{out_dir}/md-dyn-restart.xyz', atoms)

        elapsed_time = perf_counter() - s_time
        return f"Done: z_at={cv_at_str} ({elapsed_time:.1f} sec)"

    except Exception as e:
        return f"Failed: z_at={cv_at_str} Error: {e}"

## Step 4. Parallel Execution of Umbrella Sampling
Run the prepared function run_us_window in parallel using joblib. For example, if N_JOBS=4, four window calculations will run simultaneously.

In [ ]:
# ============================================================
# 4. Main Execution
# ============================================================

print(f"Start Parallel Calculation: {len(cv_restraint)} windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}")

# Parallel execution
# Format: delayed(function)(argument) for variable in list
results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window)(cv) for cv in cv_restraint
)

In [ ]:
# Display results
print("\n--- Results ---")
for res in results:
    print(res)

## Supplementary Notes

In this notebook, data collection through umbrella sampling was performed. For accurate free energy calculations, it is important that the histograms between adjacent windows sufficiently overlap along the reaction coordinate. If there is insufficient overlap, errors may occur during analysis or the calculation may not converge. If the overlap is insufficient, either weaken the spring constant `KAPPA` or add new windows in between.

## Next Step

In the next notebook [06_mbar_free_energy_en.ipynb](./06_mbar_free_energy_en.ipynb), we will proceed to the final step of computing the free energy.